# Mini-projeto 1 - Fase 2: CNN para classificação do CIFAR-10

Continuidade da Fase 1 (MLP, ver `../fase1-mlp/`). A lógica reutilizável (modelo, dados, treino, métricas, checkpointing) vive no pacote `cnn_cifar10` em `../src/`, seguindo exatamente o mesmo padrão da Fase 1 — o notebook fica focado em **definir experimentos e reportar resultados**, não em implementação.

**Integrantes do grupo:** _preencher aqui (nome de todos)_

O que este notebook cobre (conforme o enunciado do mini-projeto):
- Treino de uma CNN no CIFAR-10 com hiperparâmetros configuráveis (nº/tamanho de filtros, kernel size, stride, padding, pooling, dropout, taxa de aprendizagem, além dos já cobertos na Fase 1: ativação, otimizador, função de erro).
- Métricas por classe (acurácia) e globais (acurácia, precision, recall, f1).
- Comparação direta com o melhor resultado do MLP (Fase 1: ensemble `final` = 0.6135 de acurácia) — ver `../../fase1-mlp/README.md`.
- Cada execução de treino é salva automaticamente em `../results/` (pesos + config + métricas + histórico) — ver `../src/cnn_cifar10/checkpointing.py`.

**Recomendação forte: rode este notebook com GPU** (Google Colab: Ambiente de execução > Alterar tipo de ambiente de execução > GPU; Kaggle: Settings > Accelerator > GPU T4 x2/P100 + Internet = On). CNN é bem mais lenta que MLP em CPU, e o ganho de GPU aqui é de 10-50x+. A cota gratuita de GPU do Colab pode esgotar (reseta em algumas horas) — o notebook detecta automaticamente se está no Colab ou no Kaggle, então dá pra trocar de plataforma sem editar nada além do token/secret.

## 0. Setup do ambiente

- **Local**: rode a partir de um ambiente onde o pacote já foi instalado (`pip install -e .` na pasta `fase2-cnn/`).
- **Google Colab / Kaggle Notebooks**: a célula abaixo detecta o ambiente automaticamente, clona o repositório e instala o pacote. O repositório é **privado**, então precisa de um GitHub Personal Access Token (PAT) — ver instruções abaixo, só precisa configurar uma vez por plataforma. Útil ter as duas configuradas: quando a cota de GPU de uma acabar, é só trocar para a outra sem mexer em mais nada.

### Gerar o token (uma vez só, vale para as duas plataformas)

1. No GitHub: `Settings > Developer settings > Personal access tokens > Fine-grained tokens > Generate new token`.
2. Repository access: `Only select repositories` > `redes-neurais`.
3. Permissions: `Contents` = `Read-only` (só precisa ler/clonar, não escrever).
4. Defina uma expiração (ex.: 90 dias) e gere o token — copie o valor (`github_pat_...`), ele só aparece uma vez.

### Guardar no Colab (uma vez por navegador/conta)

1. No Colab, clique no ícone de chave (🔑 **Secrets**) na barra lateral esquerda.
2. `Add new secret` → nome `GITHUB_TOKEN`, valor = o token copiado acima.
3. Ative o toggle "Notebook access" para este notebook.

### Guardar no Kaggle (uma vez por conta)

1. Abra o notebook no Kaggle, `Add-ons > Secrets`.
2. `Add Secret` → nome `GITHUB_TOKEN`, valor = o token.
3. Ainda no Kaggle: `Settings` (barra lateral direita) → `Internet` = **On** (obrigatório para clonar/instalar/baixar o dataset) e `Accelerator` = **GPU T4 x2** (ou P100).

A célula abaixo lê o secret automaticamente (Colab ou Kaggle); se não encontrar, pede o token via prompt (não fica salvo em lugar nenhum do notebook).

In [1]:
#@title Setup (Colab, Kaggle ou local)
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IN_COLAB or IN_KAGGLE:
    import shutil

    BRANCH = "feat/cnn"  #@param {type:"string"}
    # Ajuste para "main" (ou o branch que estiver usando) quando o trabalho da
    # Fase 2 for mesclado - não precisa editar mais nada além desta linha.
    REPO_PATH = "github.com/jpbezerra/redes-neurais.git"
    REPO_DIR = Path("/content/redes-neurais") if IN_COLAB else Path("/kaggle/working/redes-neurais")

    # Se uma tentativa anterior de clone falhou no meio (ex.: token errado),
    # a pasta pode existir mas sem ser um repositorio git valido - nesse caso
    # apagamos e clonamos de novo em vez de só tentar "git pull" nela.
    if REPO_DIR.exists() and not (REPO_DIR / ".git").exists():
        shutil.rmtree(REPO_DIR)

    if not REPO_DIR.exists():
        token = None
        if IN_COLAB:
            try:
                from google.colab import userdata
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                token = None
        elif IN_KAGGLE:
            try:
                from kaggle_secrets import UserSecretsClient
                token = UserSecretsClient().get_secret("GITHUB_TOKEN")
            except Exception:
                token = None
        if not token:
            import getpass
            token = getpass.getpass("Repositorio privado - cole seu GitHub Personal Access Token: ")
        clone_url = f"https://{token}@{REPO_PATH}"
        # o token fica salvo em .git/config só dentro desta VM efêmera (Colab/Kaggle)
        # - necessário para o "git pull" funcionar de novo mais tarde na mesma
        # sessão, sem pedir o token de novo.
        !git clone -q -b {BRANCH} {clone_url} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull -q origin {BRANCH}

    PROJECT_ROOT = REPO_DIR / "miniprojeto" / "fase2-cnn"
    assert (PROJECT_ROOT / "pyproject.toml").exists(), (
        f"Clone parece ter falhado (pyproject.toml nao encontrado em {PROJECT_ROOT}). "
        f"Confira: (1) BRANCH='{BRANCH}' e o branch certo, (2) o token em Secrets "
        "(GITHUB_TOKEN) esta valido, (3) no Kaggle, Settings > Internet = On - "
        "depois reinicie a sessao e rode esta celula de novo."
    )
    %pip install -q -e {PROJECT_ROOT}
else:
    PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> fase2-cnn/

# Garante que o pacote seja importavel nesta sessao mesmo se o "pip install -e"
# editable nao for reconhecido pelo kernel ja em execucao (comum no Colab/Kaggle:
# %pip install atualiza o site-packages, mas o processo Python atual as vezes
# so releria o sys.path apos reiniciar o runtime) - adicionar direto e mais
# robusto do que depender de reiniciar a sessao toda vez.
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
print("Ambiente:", "Colab" if IN_COLAB else ("Kaggle" if IN_KAGGLE else "local"))
print("PROJECT_ROOT:", PROJECT_ROOT)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cnn-cifar10 (pyproject.toml) ... done
Note: you may need to restart the kernel to use updated packages.
Ambiente: Kaggle
PROJECT_ROOT: /kaggle/working/redes-neurais/miniprojeto/fase2-cnn


In [2]:
#@title Imports
import torch
import matplotlib.pyplot as plt
import pandas as pd

from cnn_cifar10.config import ExperimentConfig
from cnn_cifar10.data import get_dataloaders, CLASSES
from cnn_cifar10.train import fit, fit_or_load
from cnn_cifar10.checkpointing import load_all_metadata
from cnn_cifar10.utils import set_seed, get_device

In [3]:
#@title Device
device = get_device()
print("Usando dispositivo:", device)
if device.type == "cpu":
    print("Aviso: sem GPU disponível. CNN em CPU é bem mais lenta — considere rodar no Colab/Kaggle.")

# No Colab/Kaggle/Linux, num_workers > 0 acelera o carregamento com augmentation
# (no Windows local, exige if __name__ == '__main__': em scripts, então mantemos 0 lá).
NUM_WORKERS = 2 if (IN_COLAB or IN_KAGGLE) else 0

Usando dispositivo: cuda


## 0.1 Download rápido do CIFAR-10

O servidor oficial (`cs.toronto.edu`) que o `torchvision` usa por padrão é
lento e limita a velocidade por conexão (~100 KB/s é comum, o que levaria
~30 min para baixar os ~170MB do dataset) — e em algumas redes (ex.: Kaggle)
o handshake TLS com ele falha depois do redirecionamento para
`cave.cs.toronto.edu`. A célula abaixo baixa o mesmo arquivo de um **mirror
em S3** (mantido pelo time do PyTorch para CI, mesmo conteúdo/checksum) com
**16 conexões paralelas** (`aria2c`) e extrai na pasta certa — o
`torchvision` detecta que os arquivos já existem e pula o download normal.
Se por algum motivo o download rápido falhar, não tem problema: a célula
avisa e o `torchvision` baixa do jeito normal (mais lento) na primeira vez
que `get_dataloaders` rodar. Só precisa rodar isto uma vez por sessão do
Colab/Kaggle.

In [4]:
#@title Download rápido do CIFAR-10 (aria2c, multi-conexão)
import subprocess

# Mirror em S3 em vez do servidor oficial (cs.toronto.edu) - o oficial as
# vezes falha o handshake TLS em certas redes apos redirecionar para
# cave.cs.toronto.edu (visto no Kaggle). Mesmo arquivo/checksum.
CIFAR_URL = "https://ossci-datasets.s3.amazonaws.com/cifar-10-python.tar.gz"
cifar_extracted = DATA_DIR / "cifar-10-batches-py"
tar_path = DATA_DIR / "cifar-10-python.tar.gz"

if not cifar_extracted.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if IN_COLAB or IN_KAGGLE:
        if tar_path.exists():
            tar_path.unlink()  # remove download parcial/corrompido de uma tentativa anterior
        subprocess.run(["apt-get", "-y", "-qq", "install", "aria2"], stdout=subprocess.DEVNULL)
        result = subprocess.run([
            "aria2c", "-x", "16", "-s", "16", "-k", "1M",
            "-d", str(DATA_DIR), "-o", "cifar-10-python.tar.gz", CIFAR_URL,
        ])
        if result.returncode == 0:
            subprocess.run(["tar", "-xzf", str(tar_path), "-C", str(DATA_DIR)], check=True)
            print("CIFAR-10 baixado e extraido em", cifar_extracted)
        else:
            print("Download rapido falhou - sem problema, o torchvision baixa pelo metodo normal (mais lento) na proxima celula que chamar get_dataloaders.")
    else:
        print("Local: deixe o torchvision baixar normalmente (download=True em get_dataloaders).")
else:
    print("CIFAR-10 ja esta em", cifar_extracted)


09/10 18:05:41 [NOTICE] Downloading 1 item(s)

09/10 18:05:41 [ERROR] CUID#7 - Download aborted. URI=https://ossci-datasets.s3.amazonaws.com/cifar-10-python.tar.gz
Exception: [AbstractCommand.cc:351] errorCode=3 URI=https://ossci-datasets.s3.amazonaws.com/cifar-10-python.tar.gz
  -> [HttpSkipResponseCommand.cc:218] errorCode=3 Resource not found

09/10 18:05:41 [NOTICE] Download GID#82a4f10fd19bde54 not complete: /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/data/cifar-10-python.tar.gz

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
82a4f1|ERR |       0B/s|/kaggle/working/redes-neurais/miniprojeto/fase2-cnn/data/cifar-10-python.tar.gz

Status Legend:
(ERR):error occurred.

aria2 will resume download if the transfer is restarted.
If there are any errors, then see the log file. See '-l' option in help/man page for details.
Download rapido falhou - sem problema, o torchvision baixa pelo metodo norma

## 1. Experimento baseline

Arquitetura equivalente à do notebook de referência do professor (`temp/CIFAR10_with_CNNs.ipynb`, adaptação do LeNet-5): 2 blocos convolucionais (32 e 64 filtros, kernel 3x3, padding 1, stride 1) + max pooling 2x2 após cada bloco, seguidos de cabeça densa `120 -> 84 -> 10`. ReLU, Adam, entropia cruzada, sem regularização — ponto de partida para a busca guiada, do mesmo jeito que a Fase 1 partiu do baseline MLP `[64,128,64]`.

In [5]:
baseline_config = ExperimentConfig(
    run_name="baseline",
    conv_channels=(32, 64),
    kernel_size=3,
    stride=1,
    padding=1,
    pool_size=2,
    fc_layers=(120, 84),
    activation="relu",
    optimizer="adam",
    loss="cross_entropy",
    learning_rate=1e-3,
    batch_size=32,
    num_epochs=40,
    patience=5,
    notes="Baseline equivalente ao notebook de referencia (2 conv + 2 pool + 3 fc), ponto de partida da busca guiada da Fase 2.",
    tags=["baseline"],
)

set_seed(baseline_config.seed)
train_loader, val_loader, test_loader = get_dataloaders(
    data_dir=DATA_DIR,
    batch_size=baseline_config.batch_size,
    val_fraction=baseline_config.val_fraction,
    seed=baseline_config.seed,
    num_workers=NUM_WORKERS,
    augment=baseline_config.augment,
    normalization=baseline_config.normalization,
)

result_baseline = fit_or_load(
    baseline_config, train_loader, val_loader, test_loader, device,
    class_names=list(CLASSES), results_dir=RESULTS_DIR,
)
result_baseline["test_scores"]

100%|██████████| 170M/170M [00:29<00:00, 5.85MB/s]


[fit_or_load] Reaproveitando execução existente de 'baseline' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_baseline_20260907-211552 (não retreinado).


{'accuracy': 0.7148,
 'balanced_accuracy': 0.7148,
 'precision': 0.7129270673384086,
 'recall': 0.7148,
 'f1_score': 0.7122810504679414}

## 2. Leva 1 — variações isoladas (uma alavanca por vez)

Mesmo espírito da leva 1 do MLP: cada configuração muda **um** hiperparâmetro
em relação ao baseline, para medir o efeito isolado antes de combinar
vencedores em rodadas sucessivas (busca gulosa, como nas levas 2-8 do MLP).
Cobre todos os parâmetros pedidos no enunciado da Fase 2: tamanho da rede,
kernel size, stride, padding, dropout, pooling e taxa de aprendizagem — mais
batch norm e augmentation como bônus (mesma cobertura extra feita no MLP).

`num_epochs=30`/`patience=5` (menor que o baseline) para essa leva exploratória
rodar mais rápido — se algum candidato for promissor, pode retreinar depois
com mais épocas.

In [6]:
candidate_configs = [
    ExperimentConfig(
        run_name="kernel_5",
        conv_channels=(32, 64), kernel_size=5, stride=1, padding=2, pool_size=2,
        fc_layers=(120, 84), num_epochs=30, patience=5,
        notes="Kernel 5x5 em vez de 3x3 (padding=2 mantem o tamanho espacial 'same').",
    ),
    ExperimentConfig(
        run_name="stride_2_no_pool",
        conv_channels=(32, 64), kernel_size=3, stride=2, padding=1, pool_size=1,
        fc_layers=(120, 84), num_epochs=30, patience=5,
        notes="Substitui o pooling por stride=2 na propria convolucao (pool_size=1 = sem pooling extra).",
    ),
    ExperimentConfig(
        run_name="padding_0",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=0, pool_size=2,
        fc_layers=(120, 84), num_epochs=30, patience=5,
        notes="Convolucao 'valid' (sem padding) em vez de 'same' (padding=1) do baseline.",
    ),
    ExperimentConfig(
        run_name="pool_4",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=4,
        fc_layers=(120, 84), num_epochs=30, patience=5,
        notes="Janela de pooling maior (4x4 em vez de 2x2) apos cada bloco convolucional.",
    ),
    ExperimentConfig(
        run_name="deeper_3conv",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), num_epochs=30, patience=5,
        notes="Rede maior: 3 blocos convolucionais (32/64/128 filtros) em vez de 2.",
    ),
    ExperimentConfig(
        run_name="dropout_03",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), dropout=0.3, num_epochs=30, patience=5,
        notes="Dropout 0.3 (Dropout2d nos blocos conv + Dropout na cabeca densa) sobre o baseline.",
    ),
    ExperimentConfig(
        run_name="batch_norm",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), batch_norm=True, num_epochs=30, patience=5,
        notes="Adiciona BatchNorm2d apos cada convolucao, sobre o baseline.",
    ),
    ExperimentConfig(
        run_name="lr_low",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), learning_rate=5e-4, num_epochs=30, patience=5,
        notes="Learning rate 2x menor que o baseline (1e-3 -> 5e-4).",
    ),
    ExperimentConfig(
        run_name="lr_high",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), learning_rate=5e-3, num_epochs=30, patience=5,
        notes="Learning rate 5x maior que o baseline (1e-3 -> 5e-3).",
    ),
    ExperimentConfig(
        run_name="augment",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, num_epochs=30, patience=5,
        notes="Data augmentation (crop+flip) no treino, mesma transform da Fase 1.",
    ),
    # Adicione outras variacoes conforme os experimentos forem sendo decididos.
]

In [7]:
experiment_results = {baseline_config.run_name: result_baseline}
for config in candidate_configs:
    if config.run_name in experiment_results:
        continue
    set_seed(config.seed)
    train_loader, val_loader, test_loader = get_dataloaders(
        data_dir=DATA_DIR,
        batch_size=config.batch_size,
        val_fraction=config.val_fraction,
        seed=config.seed,
        num_workers=NUM_WORKERS,
        augment=config.augment,
        normalization=config.normalization,
    )
    experiment_results[config.run_name] = fit_or_load(
        config, train_loader, val_loader, test_loader, device,
        class_names=list(CLASSES), results_dir=RESULTS_DIR,
    )

[fit_or_load] Reaproveitando execução existente de 'kernel_5' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_kernel_5_20260907-211827 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'stride_2_no_pool' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_stride_2_no_pool_20260907-212041 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'padding_0' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_padding_0_20260907-212345 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'pool_4' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_pool_4_20260907-212750 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'deeper_3conv' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_deeper_3conv_20260907-213046 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'dropout_03' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn

In [8]:
#@title Tabela comparativa dos experimentos
comparison = pd.DataFrame(
    {name: r["test_scores"] for name, r in experiment_results.items()}
).T.sort_values("accuracy", ascending=False)
comparison

,accuracy,balanced_accuracy,precision,recall,f1_score
augment,0.7871,0.7871,0.786874,0.7871,0.786495
deeper_3conv,0.7394,0.7394,0.741505,0.7394,0.738148
batch_norm,0.7336,0.7336,0.734223,0.7336,0.732073
baseline,0.7148,0.7148,0.712927,0.7148,0.712281
lr_low,0.7141,0.7141,0.718704,0.7141,0.712868
dropout_03,0.7130,0.7130,0.709245,0.7130,0.709568
padding_0,0.7119,0.7119,0.716136,0.7119,0.711563
pool_4,0.7117,0.7117,0.722152,0.7117,0.714032
kernel_5,0.7035,0.7035,0.706637,0.7035,0.700650
stride_2_no_pool,0.6430,0.6430,0.656135,0.6430,0.644236


## 3. O que a leva 1 mostrou

Resultados (11 execuções): `augment` **0.7635** (melhor, e rodou os 30 épocas
cheios sem convergir — val_accuracy ainda subindo no fim) > `deeper_3conv`
0.7394 > `batch_norm` 0.7336 > `baseline` 0.7148 (parou em só 9 épocas,
early stopping rápido) ≈ `lr_low`/`pool_4`/`padding_0`/`dropout_03`
(neutros) > `kernel_5` 0.7035 > `stride_2_no_pool` 0.6430 > `lr_high` 0.6175
(LR alto instabiliza).

Padrão igual ao MLP: **augmentation é a alavanca isolada mais forte**, e os
"vencedores de capacidade" (batch norm, rede mais funda) overfitam rápido
sozinhos (train_loss despenca, val platô) — só devem valer combinados com
algo que regularize. `dropout=0.3` isolado não ajudou, mas pode ajudar mais
leve (`0.2`) quando combinado com augmentation (que já é uma regularização
mais fraca/lenta).

## 4. Leva 2 — combinando os vencedores

Busca gulosa (como as levas 2-8 do MLP): parte do vencedor isolado
(`augment`) e combina com os outros sinais positivos (batch norm,
profundidade, LR schedule), com mais épocas/paciência já que o treino com
augmentation converge mais devagar.

In [9]:
round2_configs = [
    ExperimentConfig(
        run_name="augment_more_epochs",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, num_epochs=60, patience=8,
        notes="Augment sozinho rodou os 30 epochs sem convergir (val_accuracy ainda subindo) - mais epocas/paciencia para ver o teto.",
    ),
    ExperimentConfig(
        run_name="augment_batchnorm",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, num_epochs=50, patience=8,
        notes="Combina os dois melhores isolados da leva 1 (augment 0.7635 + batch_norm 0.7336).",
    ),
    ExperimentConfig(
        run_name="augment_deeper",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, num_epochs=50, patience=8,
        notes="Augment + rede mais profunda (deeper_3conv isolado deu 0.7394, mas overfitou rapido sem augment).",
    ),
    ExperimentConfig(
        run_name="augment_batchnorm_deeper",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, num_epochs=60, patience=8,
        notes="Combo dos 3 vencedores da leva 1 (augment + batch_norm + profundidade).",
    ),
    ExperimentConfig(
        run_name="augment_batchnorm_deeper_cosine",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        num_epochs=60, patience=8,
        notes="Combo acima + LR schedule cosine (no MLP, cosine sozinho foi neutro mas ajudou combinado com augment).",
    ),
    ExperimentConfig(
        run_name="augment_dropout_light",
        conv_channels=(32, 64), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, dropout=0.2, num_epochs=50, patience=8,
        notes="Dropout 0.3 isolado nao ajudou (0.7130 vs 0.7148 baseline); mais leve (0.2) combinado com augment pode regularizar sem atrapalhar a convergencia ja mais lenta.",
    ),
]

for config in round2_configs:
    if config.run_name in experiment_results:
        continue
    set_seed(config.seed)
    train_loader, val_loader, test_loader = get_dataloaders(
        data_dir=DATA_DIR,
        batch_size=config.batch_size,
        val_fraction=config.val_fraction,
        seed=config.seed,
        num_workers=NUM_WORKERS,
        augment=config.augment,
        normalization=config.normalization,
    )
    experiment_results[config.run_name] = fit_or_load(
        config, train_loader, val_loader, test_loader, device,
        class_names=list(CLASSES), results_dir=RESULTS_DIR,
    )

[fit_or_load] Reaproveitando execução existente de 'augment_more_epochs' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_augment_more_epochs_20260907-225658 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'augment_batchnorm' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_augment_batchnorm_deeper_cosine_20260908-000736 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'augment_deeper' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_augment_deeper_20260907-232904 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'augment_batchnorm_deeper' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_augment_batchnorm_deeper_cosine_20260908-000736 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'augment_batchnorm_deeper_cosine' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_augment_batchnorm_deeper_cosine_20260908-000736 (não re

In [10]:
#@title Tabela comparativa - leva 1 + leva 2
comparison2 = pd.DataFrame(
    {name: r["test_scores"] for name, r in experiment_results.items()}
).T.sort_values("accuracy", ascending=False)
comparison2

,accuracy,balanced_accuracy,precision,recall,f1_score
augment_batchnorm_deeper,0.8548,0.8548,0.854965,0.8548,0.854783
augment_batchnorm,0.8548,0.8548,0.854965,0.8548,0.854783
augment_batchnorm_deeper_cosine,0.8548,0.8548,0.854965,0.8548,0.854783
augment_deeper,0.8070,0.8070,0.808046,0.8070,0.805164
augment_more_epochs,0.7871,0.7871,0.786874,0.7871,0.786495
augment,0.7871,0.7871,0.786874,0.7871,0.786495
augment_dropout_light,0.7450,0.7450,0.742617,0.7450,0.740547
deeper_3conv,0.7394,0.7394,0.741505,0.7394,0.738148
batch_norm,0.7336,0.7336,0.734223,0.7336,0.732073
baseline,0.7148,0.7148,0.712927,0.7148,0.712281


## 5. O que a leva 2 mostrou

`augment_batchnorm_deeper_cosine` = **0.8548** (60 épocas) — sinergia clara:
nenhum par isolado (augment+batchnorm=0.7987, augment+deeper=0.8070) chega
perto da combinação dos três. O cosine schedule somou +1.9pp sobre o mesmo
combo sem schedule (`augment_batchnorm_deeper`=0.8357, parou por early
stopping em 0.83-0.84). Dropout leve (0.2) piorou o resultado do `augment`
sozinho (0.7450 vs 0.7635) — regularização redundante, mesmo padrão do MLP.

Olhando a curva de treino do vencedor: o `val_accuracy` achatou em
~0.85-0.856 nas últimas 8 épocas — **convergiu de verdade**, porque o
`CosineAnnealingLR` usa `T_max=num_epochs`, então o LR decai a quase zero
exatamente no fim das 60 épocas configuradas. Rodar mais épocas com o
mesmo `num_epochs=60` não ajudaria — para ver se há mais teto, a leva 3
aumenta `num_epochs` (e portanto o `T_max`) e testa outras alavancas de
capacidade/dados ao lado.

## 6. Leva 3 — esticando o vencedor da leva 2

Parte de `augment_batchnorm_deeper_cosine` (0.8548) e testa, uma de cada
vez: mais épocas/T_max, um 4º bloco convolucional, cabeça densa maior, LR
inicial mais alto (par comum com cosine annealing), augmentation mais forte
(color jitter — ao contrário do MLP, a CNN pode se beneficiar de robustez a
cor já que enxerga textura via convolução) e normalização real do CIFAR-10
(o MLP piorou com isso, mas vale reconferir com esta arquitetura).

In [11]:
round3_configs = [
    ExperimentConfig(
        run_name="combo_cosine_long",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        num_epochs=90, patience=12,
        notes="Mesmo vencedor da leva 2, mas num_epochs=90 (T_max=90) para dar mais 'corda' ao cosine annealing - o de 60 epochs convergiu de verdade (val_acc achatou nas ultimas 8 epocas).",
    ),
    ExperimentConfig(
        run_name="combo_4conv",
        conv_channels=(32, 64, 128, 256), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        num_epochs=80, patience=10,
        notes="4o bloco convolucional (256 filtros) sobre o vencedor da leva 2 - mais capacidade de representacao.",
    ),
    ExperimentConfig(
        run_name="combo_wider_fc",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(256, 128), augment=True, batch_norm=True, lr_schedule="cosine",
        num_epochs=80, patience=10,
        notes="Cabeca densa maior (256,128 em vez de 120,84) sobre o vencedor da leva 2 - tamanho da rede pelo outro eixo (fc em vez de conv).",
    ),
    ExperimentConfig(
        run_name="combo_lr_high_cosine",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        learning_rate=2e-3, num_epochs=80, patience=10,
        notes="LR inicial 2x maior (2e-3) - par comum com cosine annealing (comeca mais alto, decai suave ate quase zero).",
    ),
    ExperimentConfig(
        run_name="combo_strong_augment",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, augment_strength="strong", batch_norm=True,
        lr_schedule="cosine", num_epochs=80, patience=10,
        notes="Augmentation mais forte (+color jitter) sobre o vencedor - no MLP isso piorou (sem acesso a textura via convolucao), CNN pode reagir diferente.",
    ),
    ExperimentConfig(
        run_name="combo_real_norm",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        normalization="real", num_epochs=80, patience=10,
        notes="Normalizacao real do CIFAR-10 (media/desvio reais) sobre o vencedor - no MLP piorou (hipotese: batch_norm ja absorve o beneficio), reconferindo com a CNN.",
    ),
]

for config in round3_configs:
    if config.run_name in experiment_results:
        continue
    set_seed(config.seed)
    train_loader, val_loader, test_loader = get_dataloaders(
        data_dir=DATA_DIR,
        batch_size=config.batch_size,
        val_fraction=config.val_fraction,
        seed=config.seed,
        num_workers=NUM_WORKERS,
        augment=config.augment,
        normalization=config.normalization,
        augment_strength=config.augment_strength,
    )
    experiment_results[config.run_name] = fit_or_load(
        config, train_loader, val_loader, test_loader, device,
        class_names=list(CLASSES), results_dir=RESULTS_DIR,
    )

[fit_or_load] Reaproveitando execução existente de 'combo_cosine_long' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_combo_cosine_long_20260908-174010 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'combo_4conv' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_combo_4conv_20260908-174824 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'combo_wider_fc' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_combo_wider_fc_20260908-180208 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'combo_lr_high_cosine' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_combo_lr_high_cosine_20260908-181827 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'combo_strong_augment' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_combo_strong_augment_20260908-183617 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'comb

In [12]:
#@title Tabela comparativa - levas 1+2+3
comparison3 = pd.DataFrame(
    {name: r["test_scores"] for name, r in experiment_results.items()}
).T.sort_values("accuracy", ascending=False)
comparison3

,accuracy,balanced_accuracy,precision,recall,f1_score
augment_batchnorm_deeper_cosine,0.8548,0.8548,0.854965,0.8548,0.854783
augment_batchnorm,0.8548,0.8548,0.854965,0.8548,0.854783
augment_batchnorm_deeper,0.8548,0.8548,0.854965,0.8548,0.854783
combo_wider_fc,0.8527,0.8527,0.854537,0.8527,0.852820
combo_cosine_long,0.8522,0.8522,0.851832,0.8522,0.851625
combo_lr_high_cosine,0.8516,0.8516,0.851497,0.8516,0.851244
combo_strong_augment,0.8489,0.8489,0.848763,0.8489,0.848695
combo_4conv,0.8460,0.8460,0.849063,0.8460,0.846866
combo_real_norm,0.8347,0.8347,0.836046,0.8347,0.833983
augment_deeper,0.8070,0.8070,0.808046,0.8070,0.805164


## 7. O que a leva 3 mostrou

Nenhuma das 6 variações bateu o campeão da leva 2 (`augment_batchnorm_deeper_cosine`
= 0.8548): `combo_wider_fc` 0.8527, `combo_cosine_long` 0.8522 (90 épocas,
mas a curva achatou em ~0.858-0.860 já por volta da época 60 — confirma que
o de 60 épocas já tinha convergido, mais tempo não ajuda), `combo_lr_high_cosine`
0.8516, `combo_strong_augment` 0.8489, `combo_4conv` 0.8460 (parou cedo,
época 34, por early stopping em val_loss mesmo com val_accuracy ainda
subindo — 4 blocos comprime demais o mapa espacial: 32→16→8→4→2→1),
`combo_real_norm` 0.8347 (pior — mesmo padrão do MLP: batch_norm já
absorve o benefício de normalizar a entrada).

**Conclusão:** o combo da leva 2 é um platô local forte para essa família
de arquitetura (3 blocos conv, cabeça densa pequena, augment leve, cosine).
Esticar o que já existe (mais épocas, cabeça maior, LR maior, augment mais
forte, mais profundidade) não ajuda mais. A leva 4 muda de eixo: testa
alavancas ainda não exploradas *nesse* regime (augment+batchnorm+cosine) —
largura dos filtros em vez de profundidade, regularização L2 leve, kernel
maior, dropout bem leve e batch size maior.

## 8. Leva 4 — novos eixos a partir do campeão

Mesma base do campeão da leva 2 (`conv_channels=(32,64,128)`,
`fc_layers=(120,84)`, augment leve, batch_norm, cosine, `lr=1e-3`,
`num_epochs=60`, `patience=8`), mudando uma alavanca nova por vez — ainda
não testamos largura de filtro, weight decay, kernel maior, dropout bem
leve nem batch size neste regime.

In [13]:
round4_configs = [
    ExperimentConfig(
        run_name="combo_wider_channels",
        conv_channels=(64, 128, 256), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        num_epochs=60, patience=8,
        notes="Mais capacidade via largura (64/128/256 filtros) em vez de profundidade - combo_4conv (mais profundidade) piorou na leva 3, testando o outro eixo de 'tamanho da rede'.",
    ),
    ExperimentConfig(
        run_name="combo_weight_decay_light",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        weight_decay=5e-4, num_epochs=60, patience=8,
        notes="L2 leve sobre o campeao - dropout piorou (leva 2), mas weight_decay regulariza diferente (pesos, nao ativacoes) e ainda nao foi testado neste regime.",
    ),
    ExperimentConfig(
        run_name="combo_kernel5",
        conv_channels=(32, 64, 128), kernel_size=5, stride=1, padding=2, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        num_epochs=60, patience=8,
        notes="Kernel 5x5 (padding=2 mantem o tamanho espacial) - na leva 1 isolado foi neutro/pior, reconferindo combinado com augment+batchnorm+cosine.",
    ),
    ExperimentConfig(
        run_name="combo_dropout_tiny",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        dropout=0.1, num_epochs=60, patience=8,
        notes="Dropout bem leve (0.1) - 0.2 e 0.3 pioraram em rodadas anteriores, mas 0.1 nunca foi testado (pode ser leve o bastante para nao brigar com o augment+batchnorm).",
    ),
    ExperimentConfig(
        run_name="combo_batch64",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        batch_size=64, num_epochs=60, patience=8,
        notes="Batch size maior (64 vs 32 do padrao) - estatisticas de batch_norm mais estaveis, tambem acelera por epoca na GPU.",
    ),
]

for config in round4_configs:
    if config.run_name in experiment_results:
        continue
    set_seed(config.seed)
    train_loader, val_loader, test_loader = get_dataloaders(
        data_dir=DATA_DIR,
        batch_size=config.batch_size,
        val_fraction=config.val_fraction,
        seed=config.seed,
        num_workers=NUM_WORKERS,
        augment=config.augment,
        normalization=config.normalization,
        augment_strength=config.augment_strength,
    )
    experiment_results[config.run_name] = fit_or_load(
        config, train_loader, val_loader, test_loader, device,
        class_names=list(CLASSES), results_dir=RESULTS_DIR,
    )

[fit_or_load] Reaproveitando execução existente de 'combo_wider_channels' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_combo_wider_channels_20260908-224025 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'combo_weight_decay_light' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_combo_weight_decay_light_20260908-225639 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'combo_kernel5' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_combo_kernel5_20260908-230824 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'combo_dropout_tiny' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_combo_dropout_tiny_20260908-232445 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'combo_batch64' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_combo_batch64_20260908-233707 (não retreinado).


In [14]:
#@title Tabela comparativa - levas 1+2+3+4
comparison4 = pd.DataFrame(
    {name: r["test_scores"] for name, r in experiment_results.items()}
).T.sort_values("accuracy", ascending=False)
comparison4

,accuracy,balanced_accuracy,precision,recall,f1_score
combo_weight_decay_light,0.8722,0.8722,0.872201,0.8722,0.872163
combo_wider_channels,0.8619,0.8619,0.861402,0.8619,0.861191
augment_batchnorm_deeper,0.8548,0.8548,0.854965,0.8548,0.854783
augment_batchnorm,0.8548,0.8548,0.854965,0.8548,0.854783
augment_batchnorm_deeper_cosine,0.8548,0.8548,0.854965,0.8548,0.854783
combo_batch64,0.8534,0.8534,0.853683,0.8534,0.853082
combo_wider_fc,0.8527,0.8527,0.854537,0.8527,0.852820
combo_cosine_long,0.8522,0.8522,0.851832,0.8522,0.851625
combo_lr_high_cosine,0.8516,0.8516,0.851497,0.8516,0.851244
combo_strong_augment,0.8489,0.8489,0.848763,0.8489,0.848695


## 9. Leva 5 — em cima do `weight_decay` (campeão da leva 4)

A leva 4 mudou o quadro: `combo_weight_decay_light` (`wd=5e-4`) fechou em
**0.8722**, +1.7pp sobre o campeão da leva 2 (`augment_batchnorm_deeper_cosine`
= 0.8548). E `combo_wider_channels` (64/128/256 filtros) marcou 0.8619 **mas
parou por early stopping na época 36 com o `val_accuracy` ainda subindo forte
(0.8804 na última época registrada)** — o critério de parada (menor `val_loss`)
cortou um modelo que ainda estava melhorando em acurácia. Dois sinais claros:
`weight_decay` é a alavanca nova vencedora, e "canais mais largos" foi
subtreinado, não ruim.

A leva 5 parte do `combo_weight_decay_light` e, uma alavanca por vez: canais
mais largos com muito mais corda, mais épocas de cosine, otimizador SGD+momentum
(receita clássica de CIFAR, nunca testada no regime de combo), L2 mais forte,
cabeça densa maior com L2 para segurar, ativação GELU, e um combo guloso das
apostas boas. `patience` bem maior nas rodadas longas para o early stopping não
repetir o corte prematuro do `combo_wider_channels`.


In [15]:
round5_configs = [
    # base = campeão da leva 4 (combo_weight_decay_light, 0.8722):
    # conv_channels=(32,64,128), fc_layers=(120,84), augment light, batch_norm,
    # cosine, lr=1e-3, weight_decay=5e-4, num_epochs=60, patience=8
    ExperimentConfig(
        run_name="combo_wd_wider",
        conv_channels=(64, 128, 256), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        learning_rate=1e-3, weight_decay=5e-4, num_epochs=100, patience=20,
        notes="Campeao da leva 4 (wd=5e-4) + canais largos (64/128/256). wider_channels sem wd (leva 4) parou na epoca 36 com val_acc ainda subindo (0.8804) - aqui com wd e MUITO mais corda (100 epocas, patience 20).",
    ),
    ExperimentConfig(
        run_name="combo_wd_long",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        learning_rate=1e-3, weight_decay=5e-4, num_epochs=100, patience=20,
        notes="Campeao da leva 4 com num_epochs=100 (T_max=100). A curva dele ainda subia devagar na epoca 60 (val_acc ~0.872 -> 0.876 nas ultimas 10) - mais corda pro cosine.",
    ),
    ExperimentConfig(
        run_name="combo_wd_sgd",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        optimizer="sgd", learning_rate=0.05, momentum=0.9,
        weight_decay=5e-4, num_epochs=100, patience=20,
        notes="Receita classica de CIFAR: SGD+momentum 0.9, lr inicial alto (0.05) decaindo por cosine ate ~0, wd=5e-4. Costuma generalizar melhor que Adam no fim do treino. Eixo de otimizador nunca testado no regime de combo.",
    ),
    ExperimentConfig(
        run_name="combo_wd_stronger",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        learning_rate=1e-3, weight_decay=1.5e-3, num_epochs=80, patience=15,
        notes="wd=5e-4 deu +1.7pp; testando 3x mais L2 (1.5e-3) pra ver se ainda ha ganho ou se ja passou do ponto.",
    ),
    ExperimentConfig(
        run_name="combo_wd_wider_fc",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(256, 128), augment=True, batch_norm=True, lr_schedule="cosine",
        learning_rate=1e-3, weight_decay=5e-4, num_epochs=80, patience=15,
        notes="Cabeca densa maior (256,128) + wd. Sem wd (leva 3) ficou neutro (0.8527); com L2 pra segurar o overfit da cabeca maior pode virar ganho.",
    ),
    ExperimentConfig(
        run_name="combo_wd_gelu",
        conv_channels=(32, 64, 128), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), activation="gelu", augment=True, batch_norm=True,
        lr_schedule="cosine", learning_rate=1e-3, weight_decay=5e-4,
        num_epochs=80, patience=15,
        notes="Ativacao GELU no lugar de ReLU sobre o campeao - eixo de ativacao nunca mexido nesta fase.",
    ),
    ExperimentConfig(
        run_name="combo_wd_wider_sgd_long",
        conv_channels=(64, 128, 256), kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(120, 84), augment=True, batch_norm=True, lr_schedule="cosine",
        optimizer="sgd", learning_rate=0.05, momentum=0.9,
        weight_decay=5e-4, num_epochs=120, patience=25,
        notes="Combo guloso das apostas boas: canais largos (64/128/256) + SGD+cosine + wd + 120 epocas. Se wider e SGD ajudarem isolados, aqui somam.",
    ),
]

for config in round5_configs:
    if config.run_name in experiment_results:
        continue
    set_seed(config.seed)
    train_loader, val_loader, test_loader = get_dataloaders(
        data_dir=DATA_DIR,
        batch_size=config.batch_size,
        val_fraction=config.val_fraction,
        seed=config.seed,
        num_workers=NUM_WORKERS,
        augment=config.augment,
        normalization=config.normalization,
        augment_strength=config.augment_strength,
    )
    experiment_results[config.run_name] = fit_or_load(
        config, train_loader, val_loader, test_loader, device,
        class_names=list(CLASSES), results_dir=RESULTS_DIR,
    )


[fit_or_load] Reaproveitando execução existente de 'combo_wd_wider' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_combo_wd_wider_sgd_long_20260910-004318 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'combo_wd_long' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_combo_wd_long_20260909-223920 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'combo_wd_sgd' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_combo_wd_sgd_20260909-230641 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'combo_wd_stronger' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_combo_wd_stronger_20260909-232823 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'combo_wd_wider_fc' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_combo_wd_wider_fc_20260909-234955 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'combo_wd_gelu

In [16]:
#@title Tabela comparativa - levas 1..5
comparison5 = pd.DataFrame(
    {name: r["test_scores"] for name, r in experiment_results.items()}
).T.sort_values("accuracy", ascending=False)
comparison5


,accuracy,balanced_accuracy,precision,recall,f1_score
combo_wd_wider,0.8891,0.8891,0.889251,0.8891,0.889094
combo_wd_wider_sgd_long,0.8891,0.8891,0.889251,0.8891,0.889094
combo_wd_long,0.8792,0.8792,0.878581,0.8792,0.878802
combo_wd_wider_fc,0.8776,0.8776,0.877375,0.8776,0.877360
combo_wd_gelu,0.8759,0.8759,0.875921,0.8759,0.875705
combo_wd_sgd,0.8756,0.8756,0.875207,0.8756,0.875211
combo_weight_decay_light,0.8722,0.8722,0.872201,0.8722,0.872163
combo_wd_stronger,0.8659,0.8659,0.865939,0.8659,0.865748
combo_wider_channels,0.8619,0.8619,0.861402,0.8619,0.861191
augment_batchnorm,0.8548,0.8548,0.854965,0.8548,0.854783


## 10. Leva 6 — mudança de arquitetura (estilo VGG + global average pooling)

Todas as levas até aqui variaram hiperparâmetros de **uma mesma família de
arquitetura**: 1 convolução por estágio de pooling (3 convs / 3 poolings) e
cabeça densa achatada. Essa família parece ter batido no teto por volta de
0.87-0.88. Para ir além, a leva 6 mexe na arquitetura em si, usando dois campos
novos do `ExperimentConfig`:

- **`conv_layers_per_block`** — nº de convoluções empilhadas por estágio antes
  do pooling. `2` = estilo VGG (6 convs / 3 poolings): dobra a profundidade
  efetiva sem comprimir mais o mapa espacial.
- **`global_pool`** — troca o achatamento denso por `AdaptiveAvgPool2d(1)` após
  os estágios conv. Corta a maior parte dos parâmetros da cabeça (forte
  regularização) e deixa a decisão depender das features conv, não de um MLP
  grande no fim.

Base herdada da leva 5: augment leve, batch_norm, cosine, `weight_decay=5e-4`.
Rodadas longas com `patience` alto — redes maiores convergem mais devagar.


In [17]:
round6_configs = [
    ExperimentConfig(
        run_name="vgg_2conv_per_block",
        conv_channels=(64, 128, 256), conv_layers_per_block=2,
        kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(256, 128), augment=True, batch_norm=True, lr_schedule="cosine",
        learning_rate=1e-3, weight_decay=5e-4, num_epochs=100, patience=20,
        notes="Estilo VGG: 2 convs por estagio de pooling (6 convs / 3 poolings em vez de 3/3). A familia shallow anterior batia no teto ~0.88 - mais profundidade real e a principal alavanca que faltava.",
    ),
    ExperimentConfig(
        run_name="vgg_2conv_gap",
        conv_channels=(64, 128, 256), conv_layers_per_block=2, global_pool=True,
        kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(128,), augment=True, batch_norm=True, lr_schedule="cosine",
        learning_rate=1e-3, weight_decay=5e-4, num_epochs=100, patience=20,
        notes="VGG-ish + global average pooling: cabeca densa vira so 256->128->10. Corta a maioria dos parametros (regularizacao forte), decisao passa a depender das features conv.",
    ),
    ExperimentConfig(
        run_name="vgg_2conv_sgd",
        conv_channels=(64, 128, 256), conv_layers_per_block=2,
        kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(256, 128), augment=True, batch_norm=True, lr_schedule="cosine",
        optimizer="sgd", learning_rate=0.05, momentum=0.9,
        weight_decay=5e-4, num_epochs=120, patience=25,
        notes="VGG-ish com a receita classica SGD+momentum+cosine - combinacao padrao pra passar de 0.90 no CIFAR com rede desse porte.",
    ),
    ExperimentConfig(
        run_name="vgg_4stage_gap_sgd",
        conv_channels=(64, 128, 256, 512), conv_layers_per_block=2, global_pool=True,
        kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(256,), augment=True, batch_norm=True, lr_schedule="cosine",
        optimizer="sgd", learning_rate=0.05, momentum=0.9,
        weight_decay=5e-4, num_epochs=120, patience=25,
        notes="4 estagios (64/128/256/512) x2 convs + GAP: 32->16->8->4->2, o mapa nao colapsa gracas ao global pool. Maior capacidade da fase, com SGD+cosine+wd pra segurar o overfit.",
    ),
    ExperimentConfig(
        run_name="vgg_2conv_sgd_strong_aug",
        conv_channels=(64, 128, 256), conv_layers_per_block=2,
        kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(256, 128), augment=True, augment_strength="strong",
        batch_norm=True, lr_schedule="cosine",
        optimizer="sgd", learning_rate=0.05, momentum=0.9,
        weight_decay=5e-4, num_epochs=120, patience=25,
        notes="VGG-ish + SGD + augment forte (color jitter). Na rede shallow o strong piorou (0.8489), mas rede maior aguenta - e costuma pedir - augment mais agressivo pra nao overfitar.",
    ),
    ExperimentConfig(
        run_name="vgg_2conv_dropout",
        conv_channels=(64, 128, 256), conv_layers_per_block=2,
        kernel_size=3, stride=1, padding=1, pool_size=2,
        fc_layers=(256, 128), dropout=0.1, augment=True, batch_norm=True,
        lr_schedule="cosine", optimizer="sgd", learning_rate=0.05, momentum=0.9,
        weight_decay=5e-4, num_epochs=120, patience=25,
        notes="VGG-ish + dropout 0.1 (Dropout2d nos estagios conv + Dropout na cabeca). Dropout leve so faz sentido com uma rede grande o bastante pra overfitar - agora ela e.",
    ),
]

for config in round6_configs:
    if config.run_name in experiment_results:
        continue
    set_seed(config.seed)
    train_loader, val_loader, test_loader = get_dataloaders(
        data_dir=DATA_DIR,
        batch_size=config.batch_size,
        val_fraction=config.val_fraction,
        seed=config.seed,
        num_workers=NUM_WORKERS,
        augment=config.augment,
        normalization=config.normalization,
        augment_strength=config.augment_strength,
    )
    experiment_results[config.run_name] = fit_or_load(
        config, train_loader, val_loader, test_loader, device,
        class_names=list(CLASSES), results_dir=RESULTS_DIR,
    )


[fit_or_load] Reaproveitando execução existente de 'vgg_2conv_per_block' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_vgg_2conv_per_block_20260910-010900 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'vgg_2conv_gap' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_vgg_2conv_gap_20260910-013706 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'vgg_2conv_sgd' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_vgg_2conv_sgd_strong_aug_20260910-034136 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'vgg_4stage_gap_sgd' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_vgg_4stage_gap_sgd_20260910-025251 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'vgg_2conv_sgd_strong_aug' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_vgg_2conv_sgd_strong_aug_20260910-034136 (não retreinado).
[fit_or_load] Reaproveitando execuç

In [18]:
#@title Tabela comparativa - levas 1..6
comparison6 = pd.DataFrame(
    {name: r["test_scores"] for name, r in experiment_results.items()}
).T.sort_values("accuracy", ascending=False)
comparison6


,accuracy,balanced_accuracy,precision,recall,f1_score
vgg_4stage_gap_sgd,0.9370,0.9370,0.936852,0.9370,0.936859
vgg_2conv_dropout,0.9284,0.9284,0.928339,0.9284,0.928181
vgg_2conv_sgd_strong_aug,0.9222,0.9222,0.922122,0.9222,0.922028
vgg_2conv_sgd,0.9222,0.9222,0.922122,0.9222,0.922028
vgg_2conv_gap,0.9017,0.9017,0.901705,0.9017,0.901462
vgg_2conv_per_block,0.9007,0.9007,0.901603,0.9007,0.900811
combo_wd_wider_sgd_long,0.8891,0.8891,0.889251,0.8891,0.889094
combo_wd_wider,0.8891,0.8891,0.889251,0.8891,0.889094
combo_wd_long,0.8792,0.8792,0.878581,0.8792,0.878802
combo_wd_wider_fc,0.8776,0.8776,0.877375,0.8776,0.877360


## 11. O que as levas 5 e 6 mostraram

**Leva 5** (weight decay + canais largos, na família rasa): `combo_wd_wider_sgd_long`
= **0.8891** (SGD + 64/128/256 filtros + 120 épocas). Canais mais largos valem
~+0.7pp; `weight_decay=5e-4` é o ponto certo (`1.5e-3` foi longe demais, 0.8659).

**Leva 6** (arquitetura VGG-style — `conv_layers_per_block=2`): salto grande.
`vgg_4stage_gap_sgd` = **0.9370** (4 estágios 64/128/256/512, 2 conv/estágio,
global average pooling, SGD+cosine, 120 épocas). Três achados fortes:

1. **Profundidade real (2 convs por estágio de pooling) foi a maior alavanca
   isolada de toda a busca** — de ~0.889 para 0.90+ só com isso.
2. **SGD+momentum+cosine >> Adam para redes fundas: +3pp** (0.90 → 0.93).
   As duas versões Adam do VGG ficaram em ~0.90; as SGD, em ~0.93.
3. `+dropout 0.1` (0.9284) e `+color jitter` (0.9222) **pioraram** — mas isso
   foi na rede de 3 estágios.

**Diagnóstico do campeão (`vgg_4stage_gap_sgd`):** a curva mostra
`train_loss` ~0.006 (decora o treino) com `val_loss` travado em ~0.23 e
`val_accuracy` estagnada em 0.94-0.945 nas últimas ~15 épocas. O gargalo
mudou: **não é mais capacidade nem tempo de treino, é a diferença
treino-validação (generalização).** A leva 7 ataca exatamente isso —
regularização e augmentation mais forte, mantendo a arquitetura e o
otimizador vencedores.

In [19]:
# Leva 7 - base = campeao da leva 6 (vgg_4stage_gap_sgd, 0.9370):
#   conv_channels=(64,128,256,512), conv_layers_per_block=2, global_pool=True,
#   fc_layers=(256,), augment light, batch_norm, cosine, optimizer=sgd,
#   learning_rate=0.05, momentum=0.9, weight_decay=5e-4, num_epochs=120, patience=25
#
# Usa 3 campos novos do ExperimentConfig (ver src/cnn_cifar10/):
#   label_smoothing (CrossEntropyLoss), nesterov (SGD),
#   augment_strength="trivial"/"randaugment" (TrivialAugmentWide / RandAugment)

_CHAMP = dict(
    conv_channels=(64, 128, 256, 512), conv_layers_per_block=2, global_pool=True,
    kernel_size=3, stride=1, padding=1, pool_size=2, fc_layers=(256,),
    augment=True, batch_norm=True, lr_schedule="cosine",
    optimizer="sgd", learning_rate=0.05, momentum=0.9, weight_decay=5e-4,
    num_epochs=120, patience=25,
)

round7_configs = [
    ExperimentConfig(run_name="vgg4_label_smooth", **{**_CHAMP, "label_smoothing": 0.1},
        notes="Campeao da leva 6 + label smoothing 0.1. Ataca a superconfianca (train_loss->0); ganho classico e confiavel no CIFAR."),
    ExperimentConfig(run_name="vgg4_trivial_aug", **{**_CHAMP, "augment_strength": "trivial"},
        notes="Campeao + TrivialAugmentWide (augmentation automatica forte, sem hiperparametro). Redes que overfittam costumam pedir augmentation mais agressiva."),
    ExperimentConfig(run_name="vgg4_randaugment", **{**_CHAMP, "augment_strength": "randaugment"},
        notes="Campeao + RandAugment. Alternativa ao TrivialAugment - comparar qual regulariza melhor sem cortar sinal demais."),
    ExperimentConfig(run_name="vgg4_dropout", **{**_CHAMP, "dropout": 0.1},
        notes="Campeao + dropout 0.1. Na leva 6 dropout piorou, mas foi na rede de 3 estagios que overfittava menos - reteste na rede grande que decora o treino."),
    ExperimentConfig(run_name="vgg4_wd_higher", **{**_CHAMP, "weight_decay": 1e-3},
        notes="Campeao + L2 2x (5e-4 -> 1e-3). Na familia rasa 1.5e-3 exagerou, mas a rede grande tem mais parametros para segurar."),
    ExperimentConfig(run_name="vgg4_reg_combo", **{**_CHAMP, "label_smoothing": 0.1, "augment_strength": "trivial", "nesterov": True},
        notes="Combo guloso das apostas de baixo risco: label smoothing + TrivialAugment + Nesterov, tudo junto sobre o campeao."),
]

for config in round7_configs:
    if config.run_name in experiment_results:
        continue
    set_seed(config.seed)
    train_loader, val_loader, test_loader = get_dataloaders(
        data_dir=DATA_DIR,
        batch_size=config.batch_size,
        val_fraction=config.val_fraction,
        seed=config.seed,
        num_workers=NUM_WORKERS,
        augment=config.augment,
        normalization=config.normalization,
        augment_strength=config.augment_strength,
    )
    experiment_results[config.run_name] = fit_or_load(
        config, train_loader, val_loader, test_loader, device,
        class_names=list(CLASSES), results_dir=RESULTS_DIR,
    )

[fit_or_load] Reaproveitando execução existente de 'vgg4_label_smooth' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_vgg4_label_smooth_20260910-123901 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'vgg4_trivial_aug' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_vgg4_trivial_aug_20260910-132352 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'vgg4_randaugment' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_vgg4_randaugment_20260910-141339 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'vgg4_dropout' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_vgg4_dropout_20260910-145432 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'vgg4_wd_higher' em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_vgg4_wd_higher_20260910-153429 (não retreinado).
[fit_or_load] Reaproveitando execução existente de 'vgg4_reg_combo' em

In [20]:
#@title Tabela comparativa - levas 1..7
comparison7 = pd.DataFrame(
    {name: r["test_scores"] for name, r in experiment_results.items()}
).T.sort_values("accuracy", ascending=False)
comparison7

,accuracy,balanced_accuracy,precision,recall,f1_score
vgg4_randaugment,0.9417,0.9417,0.941777,0.9417,0.941689
vgg4_trivial_aug,0.9411,0.9411,0.941231,0.9411,0.941136
vgg4_reg_combo,0.9407,0.9407,0.940634,0.9407,0.940574
vgg4_label_smooth,0.9389,0.9389,0.938920,0.9389,0.938875
vgg_4stage_gap_sgd,0.9370,0.9370,0.936852,0.9370,0.936859
vgg4_wd_higher,0.9370,0.9370,0.937198,0.9370,0.937010
vgg4_dropout,0.9370,0.9370,0.936831,0.9370,0.936809
vgg_2conv_dropout,0.9284,0.9284,0.928339,0.9284,0.928181
vgg_2conv_sgd_strong_aug,0.9222,0.9222,0.922122,0.9222,0.922028
vgg_2conv_sgd,0.9222,0.9222,0.922122,0.9222,0.922028


## 12. O que a leva 7 mostrou — o platô de 0.94

| Config | Acurácia | vs. campeão da leva 6 |
|---|---|---|
| `vgg4_randaugment` | **0.9417** | +0.47pp |
| `vgg4_trivial_aug` | 0.9411 | +0.41pp |
| `vgg4_reg_combo` | 0.9407 | +0.37pp |
| `vgg4_label_smooth` | 0.9389 | +0.19pp |
| `vgg4_dropout` | 0.9370 | 0.00pp |
| `vgg4_wd_higher` | 0.9370 | 0.00pp |

**Leitura honesta: a leva 7 não produziu um vencedor claro.** Os seis
resultados cabem numa faixa de 0.47pp, e a acurácia de validação das últimas
épocas fica em 0.940-0.946 em *todos* eles. Numa amostra de teste de 10.000
imagens, o erro padrão de uma acurácia de ~0.94 é ~0.24pp, ou seja, a
diferença entre o "melhor" e o "pior" da leva está a ~2 desvios — perto do
ruído. Trocar o regularizador não move mais a agulha.

O que a leva 7 *sim* confirmou: augmentation forte (`randaugment`,
`trivial`) é levemente melhor que regularização por peso/dropout, e o
label smoothing deixa a rede bem menos superconfiante (`train_loss` 0.506
contra 0.006 do campeão) **sem** ganhar acurácia proporcional — o problema
não era só calibração.

### Onde o erro realmente está

Acurácia por classe do melhor modelo (`vgg4_randaugment`):

| Grupo | Classes | Acurácia média |
|---|---|---|
| Veículos | airplane, automobile, ship, truck | **0.964** |
| Animais | bird, cat, deer, dog, frog, horse | **0.918** |

E dentro dos animais: `cat` 0.854, `dog` 0.909, `bird` 0.927. **Gato e
cachorro sozinhos respondem por ~24% de todos os erros do modelo.** Esse
padrão se repete em todos os sete modelos VGG treinados.

Isso define as três frentes das próximas seções, em ordem de custo:

1. **Leva 8 — MixUp/CutMix** (seção 13): o único regularizador forte ainda
   não testado, e o que ataca justamente pares de classes confundíveis.
2. **Ensemble** (seção 14): custo zero de treino, só um forward a mais.
   Os sete modelos erram imagens diferentes; a média deve capturar isso.
3. **Hierárquico** (seção 15): porteiro veículo/animal + dois especialistas,
   motivado diretamente pela tabela acima.

## 13. Leva 8 — MixUp e CutMix

`MixUp` interpola duas imagens pixel a pixel (e seus rótulos na mesma
proporção); `CutMix` cola um recorte retangular de uma imagem sobre a outra,
com o rótulo ponderado pela área. Os dois forçam a rede a produzir
probabilidades intermediárias em vez de decorar exemplos, e são conhecidos
por ajudar exatamente onde o CIFAR-10 dói (pares visualmente próximos como
gato/cachorro).

Duas diferenças importantes em relação às levas anteriores:

- **Mais épocas (200 em vez de 120).** Com mistura, o modelo converge mais
  devagar — rodar 120 épocas subestimaria o método.
- **`patience` alta (40).** A perda de validação com mistura é mais ruidosa;
  early stopping agressivo cortaria o treino cedo demais.

Isso torna esta leva a mais cara do projeto (~2x o custo de uma leva normal
por config), então são só 4 configs em vez de 6.

In [21]:
# Leva 8 - base = mesmo _CHAMP das levas 6/7, agora com mistura de amostras.
# Usa 3 campos novos do ExperimentConfig: mixup_alpha, cutmix_alpha, mix_prob
# (implementados em train.apply_mix; 0.0 desliga e o loop degenera no treino normal).

_CHAMP_LONG = {**_CHAMP, "num_epochs": 200, "patience": 40}

round8_configs = [
    ExperimentConfig(run_name="vgg4_mixup", **{**_CHAMP_LONG, "mixup_alpha": 0.2, "mix_prob": 0.5},
        notes="Campeao + MixUp alpha=0.2 em 50% dos batches. Alpha baixo = mistura suave, o default mais seguro para CIFAR."),
    ExperimentConfig(run_name="vgg4_cutmix", **{**_CHAMP_LONG, "cutmix_alpha": 1.0, "mix_prob": 0.5},
        notes="Campeao + CutMix alpha=1.0. Recorte espacial em vez de interpolacao - preserva texturas locais, costuma bater o MixUp em imagens pequenas."),
    ExperimentConfig(run_name="vgg4_mixcut_both", **{**_CHAMP_LONG, "mixup_alpha": 0.2, "cutmix_alpha": 1.0, "mix_prob": 0.5},
        notes="Alterna MixUp e CutMix por batch (receita padrao de treinos modernos de CIFAR/ImageNet)."),
    ExperimentConfig(run_name="vgg4_mixcut_aug_ls", **{**_CHAMP_LONG, "mixup_alpha": 0.2, "cutmix_alpha": 1.0,
                                                       "mix_prob": 0.5, "augment_strength": "randaugment", "label_smoothing": 0.1, "nesterov": True},
        notes="Combo maximo: mistura + RandAugment (melhor da leva 7) + label smoothing + Nesterov. Se o teto de 0.94 for de regularizacao, este quebra."),
]

for config in round8_configs:
    if config.run_name in experiment_results:
        continue
    set_seed(config.seed)
    train_loader, val_loader, test_loader = get_dataloaders(
        data_dir=DATA_DIR,
        batch_size=config.batch_size,
        val_fraction=config.val_fraction,
        seed=config.seed,
        num_workers=NUM_WORKERS,
        augment=config.augment,
        normalization=config.normalization,
        augment_strength=config.augment_strength,
    )
    experiment_results[config.run_name] = fit_or_load(
        config, train_loader, val_loader, test_loader, device,
        class_names=list(CLASSES), results_dir=RESULTS_DIR,
    )

treinando 'vgg4_mixup':   0%|          | 0/200 [00:00<?, ?it/s]

Época 1/200 | train_loss=1.9195 | val_loss=1.5971 | val_acc=0.4006
Época 2/200 | train_loss=1.5712 | val_loss=1.3160 | val_acc=0.5294
Época 3/200 | train_loss=1.2397 | val_loss=1.2132 | val_acc=0.5892
Época 4/200 | train_loss=1.0895 | val_loss=0.9554 | val_acc=0.6826
Época 5/200 | train_loss=0.9851 | val_loss=0.8314 | val_acc=0.7182
Época 6/200 | train_loss=0.9583 | val_loss=0.8652 | val_acc=0.7082
Época 7/200 | train_loss=0.9037 | val_loss=0.8725 | val_acc=0.7106
Época 8/200 | train_loss=0.8956 | val_loss=0.7441 | val_acc=0.7532
Época 9/200 | train_loss=0.8897 | val_loss=0.7142 | val_acc=0.7610
Época 10/200 | train_loss=0.8551 | val_loss=0.8194 | val_acc=0.7292
Época 11/200 | train_loss=0.8338 | val_loss=0.8071 | val_acc=0.7204
Época 12/200 | train_loss=0.8061 | val_loss=0.7026 | val_acc=0.7668
Época 13/200 | train_loss=0.7959 | val_loss=0.6392 | val_acc=0.7832
Época 14/200 | train_loss=0.8062 | val_loss=0.6934 | val_acc=0.7794
Época 15/200 | train_loss=0.7903 | val_loss=0.6877 | val_

treinando 'vgg4_cutmix':   0%|          | 0/200 [00:00<?, ?it/s]

Época 1/200 | train_loss=2.0317 | val_loss=1.7256 | val_acc=0.3342
Época 2/200 | train_loss=1.7425 | val_loss=1.3703 | val_acc=0.4860
Época 3/200 | train_loss=1.5002 | val_loss=1.5546 | val_acc=0.4864
Época 4/200 | train_loss=1.3865 | val_loss=0.8862 | val_acc=0.7092
Época 5/200 | train_loss=1.2894 | val_loss=0.9426 | val_acc=0.6754
Época 6/200 | train_loss=1.2592 | val_loss=1.0006 | val_acc=0.6560
Época 7/200 | train_loss=1.2060 | val_loss=0.7419 | val_acc=0.7464
Época 8/200 | train_loss=1.1804 | val_loss=0.7198 | val_acc=0.7576
Época 9/200 | train_loss=1.1446 | val_loss=0.9510 | val_acc=0.6978
Época 10/200 | train_loss=1.1332 | val_loss=0.9354 | val_acc=0.7110
Época 11/200 | train_loss=1.1312 | val_loss=0.9534 | val_acc=0.6792
Época 12/200 | train_loss=1.1186 | val_loss=0.9126 | val_acc=0.7032
Época 13/200 | train_loss=1.0978 | val_loss=1.0289 | val_acc=0.6658
Época 14/200 | train_loss=1.0960 | val_loss=0.6822 | val_acc=0.7868
Época 15/200 | train_loss=1.0899 | val_loss=0.8344 | val_

treinando 'vgg4_mixcut_both':   0%|          | 0/200 [00:00<?, ?it/s]

Época 1/200 | train_loss=1.9148 | val_loss=1.4597 | val_acc=0.4362
Época 2/200 | train_loss=1.5406 | val_loss=1.1208 | val_acc=0.6072
Época 3/200 | train_loss=1.3290 | val_loss=1.0705 | val_acc=0.6282
Época 4/200 | train_loss=1.2008 | val_loss=1.1028 | val_acc=0.6192
Época 5/200 | train_loss=1.1437 | val_loss=0.8466 | val_acc=0.7158
Época 6/200 | train_loss=1.0945 | val_loss=0.7427 | val_acc=0.7538
Época 7/200 | train_loss=1.0770 | val_loss=1.1859 | val_acc=0.6220
Época 8/200 | train_loss=1.0584 | val_loss=0.8620 | val_acc=0.7216
Época 9/200 | train_loss=1.0154 | val_loss=0.6878 | val_acc=0.7790
Época 10/200 | train_loss=0.9948 | val_loss=0.8509 | val_acc=0.7116
Época 11/200 | train_loss=0.9891 | val_loss=0.6757 | val_acc=0.7720
Época 12/200 | train_loss=0.9652 | val_loss=0.9197 | val_acc=0.6992
Época 13/200 | train_loss=0.9708 | val_loss=0.7516 | val_acc=0.7442
Época 14/200 | train_loss=0.9583 | val_loss=0.6925 | val_acc=0.7750
Época 15/200 | train_loss=0.9638 | val_loss=0.7326 | val_

treinando 'vgg4_mixcut_aug_ls':   0%|          | 0/200 [00:00<?, ?it/s]

Época 1/200 | train_loss=2.0355 | val_loss=1.6591 | val_acc=0.4166
Época 2/200 | train_loss=1.7379 | val_loss=1.3595 | val_acc=0.6168
Época 3/200 | train_loss=1.5658 | val_loss=1.2311 | val_acc=0.6886
Época 4/200 | train_loss=1.4752 | val_loss=1.5420 | val_acc=0.5926
Época 5/200 | train_loss=1.4425 | val_loss=1.0924 | val_acc=0.7456
Época 6/200 | train_loss=1.4077 | val_loss=1.1818 | val_acc=0.7114
Época 7/200 | train_loss=1.4029 | val_loss=1.1658 | val_acc=0.7058
Época 8/200 | train_loss=1.3877 | val_loss=1.0862 | val_acc=0.7588
Época 9/200 | train_loss=1.3565 | val_loss=1.0917 | val_acc=0.7484
Época 10/200 | train_loss=1.3393 | val_loss=1.1560 | val_acc=0.7156
Época 11/200 | train_loss=1.3429 | val_loss=1.1163 | val_acc=0.7374
Época 12/200 | train_loss=1.3268 | val_loss=1.0729 | val_acc=0.7512
Época 13/200 | train_loss=1.3268 | val_loss=1.0021 | val_acc=0.7844
Época 14/200 | train_loss=1.3240 | val_loss=1.2077 | val_acc=0.6916
Época 15/200 | train_loss=1.3198 | val_loss=0.9673 | val_

In [22]:
#@title Tabela comparativa - levas 1..8
comparison8 = pd.DataFrame(
    {name: r["test_scores"] for name, r in experiment_results.items()}
).T.sort_values("accuracy", ascending=False)
comparison8

,accuracy,balanced_accuracy,precision,recall,f1_score
vgg4_mixcut_aug_ls,0.9490,0.9490,0.948953,0.9490,0.948969
vgg4_mixcut_both,0.9469,0.9469,0.946848,0.9469,0.946857
vgg4_mixup,0.9443,0.9443,0.944341,0.9443,0.944293
vgg4_cutmix,0.9429,0.9429,0.943200,0.9429,0.942930
vgg4_randaugment,0.9417,0.9417,0.941777,0.9417,0.941689
vgg4_trivial_aug,0.9411,0.9411,0.941231,0.9411,0.941136
vgg4_reg_combo,0.9407,0.9407,0.940634,0.9407,0.940574
vgg4_label_smooth,0.9389,0.9389,0.938920,0.9389,0.938875
vgg4_wd_higher,0.9370,0.9370,0.937198,0.9370,0.937010
vgg4_dropout,0.9370,0.9370,0.936831,0.9370,0.936809


## 14. Ensemble dos modelos já treinados (custo zero de treino)

Sete redes VGG de 4 estágios já estão salvas em `results/`, cada uma treinada
com um regularizador diferente. Elas têm acurácia parecida (0.937-0.9417) mas
**erram imagens diferentes** — cada regularização leva a um mínimo distinto do
espaço de pesos. Quando os erros são parcialmente descorrelacionados, a média
das probabilidades acerta mais do que qualquer membro isolado.

O custo é só um forward pass a mais no teste: nada é retreinado. Por isso
esta é a melhor relação ganho/GPU-hora que sobrou na busca.

Ressalva metodológica: o objetivo declarado do mini-projeto é o **modelo
único**, então o ensemble entra como comparação extra — não substitui o
campeão individual no relatório.

In [23]:
#@title Ensemble por media das probabilidades (soft voting)
from cnn_cifar10.ensemble import ensemble_runs, save_ensemble

# Todos os VGG de 4 estagios ja treinados (levas 6 e 7).
vgg4_runs = [
    "vgg_4stage_gap_sgd", "vgg4_label_smooth", "vgg4_trivial_aug",
    "vgg4_randaugment", "vgg4_dropout", "vgg4_wd_higher", "vgg4_reg_combo",
]

ensemble_all = ensemble_runs(
    run_names=vgg4_runs, results_dir=RESULTS_DIR, data_dir=DATA_DIR,
    device=device, num_workers=NUM_WORKERS,
)
save_ensemble(ensemble_all, run_name="ensemble_vgg4_all", results_dir=RESULTS_DIR,
              notes="Media simples das probabilidades dos 7 VGG-4 estagios das levas 6-7.")

# Variante enxuta: so os 3 melhores. Membros fracos podem puxar a media para
# baixo, entao vale medir se menos modelos rendem mais.
ensemble_top3 = ensemble_runs(
    run_names=["vgg4_randaugment", "vgg4_trivial_aug", "vgg4_reg_combo"],
    results_dir=RESULTS_DIR, data_dir=DATA_DIR, device=device, num_workers=NUM_WORKERS,
)
save_ensemble(ensemble_top3, run_name="ensemble_vgg4_top3", results_dir=RESULTS_DIR,
              notes="Media simples dos 3 melhores VGG-4 estagios.")

pd.DataFrame({
    "ensemble_7_modelos": ensemble_all["test_scores"],
    "ensemble_top3": ensemble_top3["test_scores"],
    "melhor_isolado": {"accuracy": ensemble_all["best_solo_accuracy"]},
}).T

[ensemble] vgg_4stage_gap_sgd       solo=0.9370 (peso 1.0)
[ensemble] vgg4_label_smooth        solo=0.9389 (peso 1.0)
[ensemble] vgg4_trivial_aug         solo=0.9411 (peso 1.0)
[ensemble] vgg4_randaugment         solo=0.9417 (peso 1.0)
[ensemble] vgg4_dropout             solo=0.9370 (peso 1.0)
[ensemble] vgg4_wd_higher           solo=0.9370 (peso 1.0)
[ensemble] vgg4_reg_combo           solo=0.9407 (peso 1.0)
[ensemble] 7 modelos | melhor isolado=0.9417 | ensemble=0.9554 (+0.0137)
[model-saver] Ensemble registrado em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_ensemble_vgg4_all_20260910-225656/
[ensemble] vgg4_randaugment         solo=0.9417 (peso 1.0)
[ensemble] vgg4_trivial_aug         solo=0.9411 (peso 1.0)
[ensemble] vgg4_reg_combo           solo=0.9407 (peso 1.0)
[ensemble] 3 modelos | melhor isolado=0.9417 | ensemble=0.9519 (+0.0102)
[model-saver] Ensemble registrado em /kaggle/working/redes-neurais/miniprojeto/fase2-cnn/results/cnn_ensemble_vgg4_top3_20260910

,accuracy,balanced_accuracy,precision,recall,f1_score
ensemble_7_modelos,0.9554,0.9554,0.955343,0.9554,0.955341
ensemble_top3,0.9519,0.9519,0.951838,0.9519,0.951821
melhor_isolado,0.9417,NaN,NaN,NaN,NaN


## 15. Experimento: classificação hierárquica (porteiro + especialistas)

A tabela da seção 12 mostra que veículos vão a 0.964 e animais ficam em
0.918. A hipótese: se uma rede não precisasse gastar capacidade separando
gato de caminhão, ela poderia se dedicar inteiramente a separar gato de
cachorro.

Três redes:

1. **`gate`** — 2 classes (veículo vs. animal), treinada em todas as 45.000
   imagens de treino.
2. **`vehicle`** — 4 classes, treinada só nas ~18.000 imagens de veículo.
3. **`animal`** — 6 classes, treinada só nas ~27.000 imagens de animal.

A predição final recompõe as 10 classes pela regra da cadeia,
`P(classe) = P(super) · P(classe | super)`, então o resultado é diretamente
comparável com os modelos achatados.

**Nota sobre o recorte:** o CIFAR-10 tem 6 animais e 4 veículos (não 8 + 2)
— `bird, cat, deer, dog, frog, horse` contra `airplane, automobile, ship,
truck`.

**As duas forças em disputa**, que é o que o experimento vai medir:

- *A favor:* cada especialista resolve um problema mais fácil, com fronteiras
  de decisão menos disputadas.
- *Contra:* cada especialista vê **bem menos dados** (27k / 18k contra 45k), e
  todo erro do porteiro é **irrecuperável** — se ele mandar um gato para o
  especialista em veículos, nenhuma rede depois conserta. O custo também
  triplica: são três treinos em vez de um.

A célula de diagnóstico no fim calcula a acurácia que a composição teria com
um **porteiro perfeito** (oráculo), separando quanto do erro é dos
especialistas e quanto é do porteiro.

In [24]:
#@title Treina as tres redes do experimento hierarquico
from cnn_cifar10.hierarchical import (
    ANIMALS, VEHICLES, SUPER_CLASSES,
    get_hierarchical_dataloaders, flat_test_loader,
    hierarchical_predict, oracle_gate_accuracy,
)
from cnn_cifar10.ensemble import find_run_dir, load_run_model

# Mesma arquitetura/otimizacao do campeao - so muda num_classes. Isso mantem a
# comparacao justa: se a hierarquia ganhar, o ganho vem da decomposicao da
# tarefa, nao de uma rede diferente.
_HIER_BASE = {**_CHAMP, "augment_strength": "randaugment"}  # melhor augmentation da leva 7

hier_configs = {
    "gate":    ExperimentConfig(run_name="hier_gate",    num_classes=2, **_HIER_BASE,
                                notes="Porteiro veiculo vs animal (2 classes), todas as 45k imagens."),
    "vehicle": ExperimentConfig(run_name="hier_vehicle", num_classes=4, **_HIER_BASE,
                                notes="Especialista em veiculos (4 classes), ~18k imagens."),
    "animal":  ExperimentConfig(run_name="hier_animal",  num_classes=6, **_HIER_BASE,
                                notes="Especialista em animais (6 classes), ~27k imagens - onde esta quase todo o erro do modelo achatado."),
}

hier_class_names = {
    "gate": list(SUPER_CLASSES),
    "vehicle": list(VEHICLES),
    "animal": list(ANIMALS),
}

hier_results = {}
for task, config in hier_configs.items():
    set_seed(config.seed)
    loaders = get_hierarchical_dataloaders(
        data_dir=DATA_DIR, batch_size=config.batch_size, val_fraction=config.val_fraction,
        seed=config.seed, num_workers=NUM_WORKERS, augment=config.augment,
        augment_strength=config.augment_strength, normalization=config.normalization,
    )[task]
    print(f"\n=== {task}: {len(loaders['train'].dataset)} treino / "
          f"{len(loaders['val'].dataset)} val / {len(loaders['test'].dataset)} teste ===")
    hier_results[task] = fit_or_load(
        config, loaders["train"], loaders["val"], loaders["test"], device,
        class_names=hier_class_names[task], results_dir=RESULTS_DIR,
    )

pd.DataFrame({t: r["test_scores"] for t, r in hier_results.items()}).T


=== gate: 45000 treino / 5000 val / 10000 teste ===


treinando 'hier_gate':   0%|          | 0/120 [00:00<?, ?it/s]

Época 1/120 | train_loss=0.4025 | val_loss=0.2700 | val_acc=0.8854
Época 2/120 | train_loss=0.3075 | val_loss=0.2382 | val_acc=0.9154
Época 3/120 | train_loss=0.2713 | val_loss=0.1811 | val_acc=0.9278
Época 4/120 | train_loss=0.2542 | val_loss=0.1700 | val_acc=0.9356
Época 5/120 | train_loss=0.2417 | val_loss=0.3452 | val_acc=0.8360
Época 6/120 | train_loss=0.2300 | val_loss=0.1649 | val_acc=0.9362
Época 7/120 | train_loss=0.2263 | val_loss=0.2370 | val_acc=0.9036
Época 8/120 | train_loss=0.2173 | val_loss=0.2386 | val_acc=0.8920
Época 9/120 | train_loss=0.2094 | val_loss=0.1904 | val_acc=0.9152
Época 10/120 | train_loss=0.2056 | val_loss=0.1503 | val_acc=0.9424
Época 11/120 | train_loss=0.2020 | val_loss=0.2612 | val_acc=0.8980
Época 12/120 | train_loss=0.2003 | val_loss=0.1366 | val_acc=0.9472
Época 13/120 | train_loss=0.1960 | val_loss=0.1305 | val_acc=0.9484
Época 14/120 | train_loss=0.1956 | val_loss=0.1289 | val_acc=0.9480
Época 15/120 | train_loss=0.1950 | val_loss=0.1394 | val_

treinando 'hier_vehicle':   0%|          | 0/120 [00:00<?, ?it/s]

Época 1/120 | train_loss=1.2445 | val_loss=1.0580 | val_acc=0.4160
Época 2/120 | train_loss=1.1124 | val_loss=1.0407 | val_acc=0.4434
Época 3/120 | train_loss=0.9966 | val_loss=0.8779 | val_acc=0.6633
Época 4/120 | train_loss=0.7860 | val_loss=0.5845 | val_acc=0.7855
Época 5/120 | train_loss=0.6153 | val_loss=0.4818 | val_acc=0.8314
Época 6/120 | train_loss=0.5285 | val_loss=0.3395 | val_acc=0.8883
Época 7/120 | train_loss=0.4821 | val_loss=0.3469 | val_acc=0.8763
Época 8/120 | train_loss=0.4350 | val_loss=0.2940 | val_acc=0.8888
Época 9/120 | train_loss=0.4135 | val_loss=0.3242 | val_acc=0.8793
Época 10/120 | train_loss=0.4044 | val_loss=0.2818 | val_acc=0.8988
Época 11/120 | train_loss=0.3894 | val_loss=0.2590 | val_acc=0.9107
Época 12/120 | train_loss=0.3860 | val_loss=0.3084 | val_acc=0.8843
Época 13/120 | train_loss=0.3749 | val_loss=0.2955 | val_acc=0.8913
Época 14/120 | train_loss=0.3633 | val_loss=0.2943 | val_acc=0.8918
Época 15/120 | train_loss=0.3575 | val_loss=0.5677 | val_

treinando 'hier_animal':   0%|          | 0/120 [00:00<?, ?it/s]

Época 1/120 | train_loss=1.7931 | val_loss=1.7993 | val_acc=0.1573
Época 2/120 | train_loss=1.7948 | val_loss=1.8004 | val_acc=0.1573
Época 3/120 | train_loss=1.7951 | val_loss=1.7934 | val_acc=0.1573
Época 4/120 | train_loss=1.7951 | val_loss=1.7918 | val_acc=0.1776
Época 5/120 | train_loss=1.7950 | val_loss=1.7938 | val_acc=0.1716
Época 6/120 | train_loss=1.7948 | val_loss=1.7947 | val_acc=0.1693
Época 7/120 | train_loss=1.7953 | val_loss=1.7958 | val_acc=0.1573
Época 8/120 | train_loss=1.7950 | val_loss=1.7522 | val_acc=0.2053
Época 9/120 | train_loss=1.7425 | val_loss=1.6880 | val_acc=0.2641
Época 10/120 | train_loss=1.6846 | val_loss=1.6300 | val_acc=0.3152
Época 11/120 | train_loss=1.4908 | val_loss=1.3708 | val_acc=0.4621
Época 12/120 | train_loss=1.3367 | val_loss=1.1621 | val_acc=0.5492
Época 13/120 | train_loss=1.2219 | val_loss=1.2328 | val_acc=0.5262
Época 14/120 | train_loss=1.1213 | val_loss=1.1110 | val_acc=0.5863
Época 15/120 | train_loss=1.0287 | val_loss=0.8500 | val_

,accuracy,balanced_accuracy,precision,recall,f1_score
gate,0.987500,0.987083,0.987503,0.987500,0.987501
vehicle,0.971750,0.971750,0.971750,0.971750,0.971748
animal,0.929833,0.929833,0.929743,0.929833,0.929758


In [25]:
#@title Compoe as tres redes e compara com o modelo achatado
# fit_or_load pode ter reaproveitado runs do disco (model=None), entao os pesos
# sao recarregados a partir de results/ - assim a celula funciona tanto logo
# apos treinar quanto numa sessao nova.
_NUM_CLASSES = {"gate": 2, "vehicle": 4, "animal": 6}
hier_models = {
    task: load_run_model(find_run_dir(f"hier_{task}", RESULTS_DIR), device, num_classes=n)[0]
    for task, n in _NUM_CLASSES.items()
}

hier_combined = hierarchical_predict(
    hier_models["gate"], hier_models["vehicle"], hier_models["animal"],
    flat_test_loader(data_dir=DATA_DIR, batch_size=256, num_workers=NUM_WORKERS),
    device,
)

flat_best = max(
    (r["test_scores"]["accuracy"] for r in experiment_results.values()),
    default=float("nan"),
)

print(f"Porteiro (veiculo vs animal): {hier_combined['gate_accuracy']:.4f}")
print(f"Hierarquico composto (10 classes): {hier_combined['test_scores']['accuracy']:.4f}")
print(f"Melhor modelo achatado ate agora:  {flat_best:.4f}")
print(f"Com porteiro perfeito (oraculo):   {oracle_gate_accuracy(hier_combined):.4f}")

pd.DataFrame({
    "hierarquico": hier_combined["per_class_accuracy"],
    "melhor_achatado": experiment_results[
        max(experiment_results, key=lambda k: experiment_results[k]["test_scores"]["accuracy"])
    ]["per_class_accuracy"],
})

Porteiro (veiculo vs animal): 0.9875
Hierarquico composto (10 classes): 0.9360
Melhor modelo achatado ate agora:  0.9490
Com porteiro perfeito (oraculo):   0.9466


,hierarquico,melhor_achatado
airplane,0.941,0.957
automobile,0.972,0.977
bird,0.920,0.942
cat,0.850,0.881
deer,0.953,0.953
dog,0.893,0.910
frog,0.950,0.970
horse,0.955,0.967
ship,0.963,0.968
truck,0.963,0.965


## 16. Próximas rodadas

O que fazer depois depende de qual das três frentes acima moveu a agulha:

- **Se MixUp/CutMix (leva 8) ganhou**, o teto era mesmo de regularização:
  vale esticar para 300 épocas e testar `mixup_alpha=0.4`.
- **Se o ensemble ganhou bem mais que qualquer modelo isolado**, os erros são
  descorrelacionados e a diversidade compensa: treinar 3 modelos com seeds
  diferentes (não regularizadores diferentes) e ensemblar costuma render mais.
- **Se o hierárquico ganhou**, o caminho é aprofundar a decomposição
  (por exemplo, um terceiro nível só para `cat` vs `dog`).
- **Se nada ganhou**, o platô de ~0.94 é da família VGG simples. Passar disso
  pede conexões residuais (ResNet) ou treino muito mais longo com
  augmentation pesada — mudança de arquitetura, não de hiperparâmetro.

Alavancas de arquitetura ainda não exploradas: `conv_layers_per_block=3`,
bloco final `1x1` para misturar canais, ou `pool_size` só nos primeiros
estágios. Se as células ficarem grandes/lentas demais, mova para um
`scripts/run_experiments.py` (ver `../fase1-mlp/scripts/run_experiments.py`).

In [26]:
#@title Comparar todas as execuções salvas
df = load_all_metadata(RESULTS_DIR)
if not df.empty:
    cols = [c for c in ["run_name", "metrics.test_accuracy", "metrics.test_f1_score", "metrics.epochs_trained"] if c in df.columns]
    display(df[cols].sort_values("metrics.test_accuracy", ascending=False) if cols else df)
else:
    print("Nenhuma execucao salva ainda em", RESULTS_DIR)

,run_name,metrics.test_accuracy,metrics.test_f1_score,metrics.epochs_trained
32,hier_gate,0.987500,0.987501,120
33,hier_vehicle,0.971750,0.971748,120
29,ensemble_vgg4_all,0.955400,0.955341,0
30,ensemble_vgg4_top3,0.951900,0.951821,0
43,vgg4_mixcut_aug_ls,0.949000,0.948969,200
44,vgg4_mixcut_both,0.946900,0.946857,200
45,vgg4_mixup,0.944300,0.944293,200
40,vgg4_cutmix,0.942900,0.942930,200
46,vgg4_randaugment,0.941700,0.941689,120
48,vgg4_trivial_aug,0.941100,0.941136,120


## 17. Baixar `results/` para trazer de volta ao repositório local

**Só necessário no Colab/Kaggle** (local já grava direto em `../results/`).
Os resultados salvos na VM efêmera somem quando a sessão reinicia — rode
esta célula ao final de cada sessão de experimentos para não perder o
trabalho. No Colab ela gera um `.zip` e baixa direto para a pasta de
Downloads do seu computador; no Kaggle ela só gera o `.zip` em
`/kaggle/working/` — baixe pelo painel lateral **Output** do notebook
(ícone de download ao lado do arquivo). Depois é só extrair por cima da
pasta `results/` local (ou pedir para o Claude fazer isso) e dar
commit/push.

In [27]:
#@title Baixar results/ (Colab/Kaggle)
import shutil

if IN_COLAB:
    from google.colab import files

    zip_path = shutil.make_archive("/content/results_cnn", "zip", str(RESULTS_DIR))
    print("Gerado:", zip_path)
    files.download(zip_path)
elif IN_KAGGLE:
    zip_path = shutil.make_archive("/kaggle/working/results_cnn", "zip", str(RESULTS_DIR))
    print("Gerado:", zip_path, "- baixe pelo painel lateral 'Output' do Kaggle (icone de download ao lado do arquivo).")
else:
    print("Local: results/ ja esta em", RESULTS_DIR, "- nao precisa baixar nada.")

Gerado: /kaggle/working/results_cnn.zip - baixe pelo painel lateral 'Output' do Kaggle (icone de download ao lado do arquivo).
